[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/02.%20Parte%202/12.%20Clase%2012/12Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=02.%20Parte%202%2F12.%20Clase%2012%2F12Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels", "cvxpy"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 12: Optimización avanzada de portafolios y estrategias de opciones

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Aplicar **regularización L₂** (Ridge) y **L₁** (LASSO) al portafolio de Markowitz con CVXPY.
- Formular restricciones de **tracking error** como SOCP (Boyd & Vandenberghe, 2004, §4.3.1).
- Comparar fronteras eficientes: estándar, regularizada L₂, y con tracking error.
- Construir y graficar **estrategias de opciones**: bull spread, bear spread, straddle.

---

## Introducción teórica

### Extensiones del modelo de Markowitz

El QP estándar de Markowitz (Clase 4/11) asume que la covarianza y la media son conocidas exactamente. En la práctica:
- Los pesos óptimos son **inestables** ante errores de estimación (Michaud, 1989)
- Se necesitan **restricciones adicionales** (tracking error, límites sectoriales)
- La **regularización** estabiliza los pesos

Estas extensiones mantienen la convexidad del problema (Boyd & Vandenberghe, 2004, §4.4, §6.3).

## 1. Datos y estimadores robustos

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.covariance as skcov
import statsmodels.api as sm

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 4)

In [ ]:
tickers = ["AAPL", "AMZN", "MSFT", "KO", "AA", "NVDA"]
data = yf.download(tickers, start="2025-01-01", end="2025-03-27",
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()
daily_returns = np.log(closes / closes.shift(1)).dropna()

# Estimadores robustos
huber = sm.robust.scale.Huber()
mu = np.array([huber(daily_returns[col].values)[0] for col in daily_returns.columns])
Sigma = skcov.LedoitWolf().fit(daily_returns).covariance_
n = len(tickers)

print(f"Activos: {tickers}")
print(f"Observaciones: {len(daily_returns)}")

---
## 2. Frontera eficiente estándar (referencia)

In [ ]:
def efficient_frontier(mu_vec, Sigma, n_points=150, constraints_extra=None):
    """Frontera eficiente con CVXPY. Acepta restricciones adicionales."""
    n = len(mu_vec)
    w = cp.Variable(n)
    target = cp.Parameter()
    risk = cp.quad_form(w, Sigma)
    cons = [mu_vec @ w == target, cp.sum(w) == 1, w >= 0]
    if constraints_extra:
        cons += constraints_extra(w)
    prob = cp.Problem(cp.Minimize(risk), cons)
    means, stds = [], []
    for r in np.linspace(mu_vec.min(), mu_vec.max(), n_points):
        target.value = r
        prob.solve(solver=cp.ECOS, warm_start=True)
        if prob.status == "optimal":
            means.append(252 * r)
            stds.append(np.sqrt(252 * risk.value))
    return np.array(means), np.array(stds)

ef_means, ef_stds = efficient_frontier(mu, Sigma)
print(f"Frontera estándar: {len(ef_means)} puntos")

---
## 3. Regularización L₂ (Ridge / Tikhonov)

Agregar $\gamma \|\mathbf{w}\|_2^2$ al objetivo estabiliza los pesos (Boyd & Vandenberghe, 2004, §6.3.2):

$$
\min_{\mathbf{w}} \quad \mathbf{w}^\top \Sigma \, \mathbf{w} + \gamma \|\mathbf{w}\|_2^2 \qquad \text{s.a.} \quad \boldsymbol{\mu}^\top \mathbf{w} = \mu^*, \; \sum w_i = 1, \; w_i \geq 0
$$

Equivale a reemplazar $\Sigma$ por $\Sigma + \gamma I$ en el QP. El efecto:
- Mejora el **número de condición** de la Hessiana
- Penaliza pesos grandes → más **diversificación**
- Sigue siendo un QP convexo ($\Sigma + \gamma I \succ 0$)

In [ ]:
gammas = [0.0001, 0.001, 0.01]

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(ef_stds, ef_means, "b-", linewidth=2, label="Estándar (γ=0)")

for gamma, color, ls in zip(gammas, ["tab:orange", "tab:green", "tab:red"], ["--", "-.", ":"]):
    Sigma_reg = Sigma + gamma * np.eye(n)
    means_r, stds_r = efficient_frontier(mu, Sigma_reg)
    ax.plot(stds_r, means_r, color=color, linewidth=2, linestyle=ls,
            label=f"Ridge γ={gamma}")

ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Efecto de la regularización L₂ en la frontera eficiente")
ax.legend()
plt.tight_layout()

---
## 4. Restricción de tracking error (SOCP)

El **tracking error** respecto a un benchmark $\mathbf{w}_b$ es:

$$
\text{TE} = \sqrt{(\mathbf{w} - \mathbf{w}_b)^\top \Sigma (\mathbf{w} - \mathbf{w}_b)}
$$

La restricción $\text{TE} \leq \tau$ define un **elipsoide** en el espacio de pesos, que es un conjunto convexo. El problema resultante es un SOCP (Boyd & Vandenberghe, 2004, §4.3.1).

In [ ]:
# Benchmark: pesos iguales
w_bench = np.ones(n) / n
tau = 0.003  # Tracking error máximo diario

def te_constraint(w):
    L = np.linalg.cholesky(Sigma + 1e-8 * np.eye(n))
    return [cp.norm(L @ (w - w_bench), 2) <= tau]

means_te, stds_te = efficient_frontier(mu, Sigma, constraints_extra=te_constraint)
print(f"Frontera con TE ≤ {tau}: {len(means_te)} puntos")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(ef_stds, ef_means, "b-", linewidth=2, label="Sin restricción")
if len(means_te) > 0:
    ax.plot(stds_te, means_te, "r--", linewidth=2, label=f"Con TE ≤ {tau}")

# Benchmark
ret_b = 252 * mu @ w_bench
risk_b = np.sqrt(252 * w_bench @ Sigma @ w_bench)
ax.scatter(risk_b, ret_b, marker="D", s=150, color="gold", zorder=5,
           edgecolors="black", label="Benchmark (pesos iguales)")

ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Frontera eficiente con restricción de tracking error")
ax.legend()
plt.tight_layout()

---
## 5. Comparación de pesos óptimos

Comparamos los pesos del portafolio de máximo rendimiento bajo cada esquema:

In [ ]:
# Max rendimiento factible para cada frontera
def max_sharpe_weights(mu_vec, Sigma_mat, rf=0.0, extra_cons=None):
    nn = len(mu_vec)
    excess = mu_vec - rf
    y = cp.Variable(nn)
    kappa = cp.Variable(nonneg=True)
    cons = [excess @ y == 1, cp.sum(y) == kappa, y >= 0]
    if extra_cons:
        cons += extra_cons(y)
    prob = cp.Problem(cp.Minimize(cp.quad_form(y, Sigma_mat)), cons)
    prob.solve(solver=cp.ECOS)
    if prob.status == "optimal":
        return y.value / kappa.value
    return np.ones(nn) / nn

w_std = max_sharpe_weights(mu, Sigma)
w_ridge = max_sharpe_weights(mu, Sigma + 0.001 * np.eye(n))

comp = pd.DataFrame({"Estándar": w_std, "Ridge (γ=0.001)": w_ridge}, index=tickers)
comp["Diferencia"] = comp["Ridge (γ=0.001)"] - comp["Estándar"]
comp

In [ ]:
comp[["Estándar", "Ridge (γ=0.001)"]].plot(kind="bar", figsize=(10, 5))
plt.title("Pesos óptimos: estándar vs. regularizado L₂")
plt.ylabel("Peso")
plt.xticks(rotation=0)
plt.tight_layout()

---
## 6. Estrategias de opciones

### Funciones de payoff y P&L

Reutilizamos las funciones vectorizadas de la Clase 3 para construir **estrategias combinadas** de opciones.

In [ ]:
def option_payoff(ST, K, option_type="call"):
    if option_type == "call":
        return np.maximum(ST - K, 0)
    else:
        return np.maximum(K - ST, 0)

def option_pnl(ST, K, premium, option_type="call", position="buyer"):
    payoff = option_payoff(ST, K, option_type)
    return payoff - premium if position == "buyer" else premium - payoff

ST = np.linspace(20, 50, 500)

### 6.1 Bull Call Spread (diferencial alcista)

Comprar una call con strike bajo $K_1$ y vender una call con strike alto $K_2 > K_1$. **Beneficio limitado** pero costo menor que una call sola.

$$
\text{PnL} = \max(S_T - K_1, 0) - c_1 - [\max(S_T - K_2, 0) - c_2]
$$

**Ejemplo (Hull)**: Comprar call $K_1 = 30$ por \$3, vender call $K_2 = 35$ por \$1. Costo neto: \$2.

In [ ]:
K1, c1 = 30, 3  # Call comprada
K2, c2 = 35, 1  # Call vendida

pnl_bull = option_pnl(ST, K1, c1, "call", "buyer") + option_pnl(ST, K2, c2, "call", "seller")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ST, pnl_bull, linewidth=2, color="tab:blue")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.fill_between(ST, pnl_bull, 0, where=(pnl_bull > 0), alpha=0.15, color="green")
ax.fill_between(ST, pnl_bull, 0, where=(pnl_bull < 0), alpha=0.15, color="red")
ax.axvline(K1, color="gray", linestyle=":", alpha=0.5, label=f"K₁ = {K1}")
ax.axvline(K2, color="gray", linestyle=":", alpha=0.5, label=f"K₂ = {K2}")
ax.set_title(f"Bull Call Spread (K₁={K1}, c₁={c1} | K₂={K2}, c₂={c2})")
ax.set_xlabel("$S_T$")
ax.set_ylabel("P&L")
ax.legend()
plt.tight_layout()

### 6.2 Bear Put Spread (diferencial bajista)

Comprar put con strike alto $K_2$ y vender put con strike bajo $K_1 < K_2$. Beneficia si el precio **baja**.

In [ ]:
K1_p, p1 = 30, 1  # Put vendida
K2_p, p2 = 35, 4  # Put comprada

pnl_bear = option_pnl(ST, K2_p, p2, "put", "buyer") + option_pnl(ST, K1_p, p1, "put", "seller")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ST, pnl_bear, linewidth=2, color="tab:red")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.fill_between(ST, pnl_bear, 0, where=(pnl_bear > 0), alpha=0.15, color="green")
ax.fill_between(ST, pnl_bear, 0, where=(pnl_bear < 0), alpha=0.15, color="red")
ax.set_title(f"Bear Put Spread (K₁={K1_p}, p₁={p1} | K₂={K2_p}, p₂={p2})")
ax.set_xlabel("$S_T$")
ax.set_ylabel("P&L")
plt.tight_layout()

### 6.3 Straddle (cono)

Comprar call **y** put con el mismo strike $K$. Beneficia si el precio se mueve **mucho** en cualquier dirección (alta volatilidad).

In [ ]:
K_s = 35
c_s, p_s = 3, 3  # Primas iguales (ATM)

pnl_straddle = option_pnl(ST, K_s, c_s, "call", "buyer") + option_pnl(ST, K_s, p_s, "put", "buyer")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ST, pnl_straddle, linewidth=2, color="tab:purple")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.fill_between(ST, pnl_straddle, 0, where=(pnl_straddle > 0), alpha=0.15, color="green")
ax.fill_between(ST, pnl_straddle, 0, where=(pnl_straddle < 0), alpha=0.15, color="red")
ax.axvline(K_s, color="gray", linestyle=":", alpha=0.5, label=f"K = {K_s}")
ax.set_title(f"Straddle (K={K_s}, c={c_s}, p={p_s})")
ax.set_xlabel("$S_T$")
ax.set_ylabel("P&L")
ax.legend()
plt.tight_layout()

### 6.4 Resumen de estrategias

| Estrategia | Composición | Visión del mercado | Ganancia máx. | Pérdida máx. |
|-----------|-------------|-------------------|:---:|:---:|
| **Bull Call Spread** | Long call K₁ + short call K₂ | Moderadamente alcista | K₂ - K₁ - costo | Costo neto |
| **Bear Put Spread** | Long put K₂ + short put K₁ | Moderadamente bajista | K₂ - K₁ - costo | Costo neto |
| **Straddle** | Long call K + long put K | Alta volatilidad | Ilimitada | c + p |

---

## Navegación del curso

← **Anterior**: Clase 11: Frontera eficiente con CVXPY  
→ **Siguiente**: Clase 13: Portafolio con bono (MC vs Markowitz)

---

## 7. Referencias bibliográficas

- **Boyd, S. & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. — §4.3.1 (SOCP), §6.3 (regularización L₁/L₂).
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson. — Cap. 12: Trading Strategies Involving Options.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **Michaud, R. O.** (1989). The Markowitz Optimization Enigma. *Financial Analysts Journal*, 45(1), 31–42.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.